## **Trabajo Práctico Final: Telco Churn**
____
__Universidad Tecnológica Nacional, Buenos Aires__\
__Ingeniería Industrial__\
__Ciencia de Datos - Curso I5571__
____
**Notebook 2 de 3: Manipulación de datos**

**Grupo 6**

| Legajo | Integrante |
|---|---|
| 1764111 | Guillermo Inostroza |
| 1772752 | Erik Yuji Tanaka |
| 1593973 | Iñaki Hegoburu |
| 2042873 | Julián Longo |

# 1. Objetivo y alcance

Este notebook reúne las manipulaciones que hicimos sobre los datos para aplicar el modelo. Parte del CSV original (`data/telco_churn_clusterai.csv`), aplica las transformaciones que decidió el EDA y guarda un dataset listo para el notebook de machine learning. Como tiene que correr sin inconvenientes en otra computadora, cada paso termina en controles que imprimen `OK` o detienen la ejecución con un mensaje que dice qué falló.

El trabajo tiene tres notebooks, que se ejecutan en este orden:

| Orden | Notebook | Qué deja |
|---|---|---|
| 1 | `clusterai_guillermo_inostroza_eda.ipynb`: exploración y 18 decisiones | `data/particion_train_test.csv` |
| 2 | `clusterai_guillermo_inostroza_manipulacion.ipynb` (este) | `data/telco_manipulado.csv` |
| 3 | `clusterai_guillermo_inostroza_machine_learning.ipynb`: pipeline, validación cruzada y test | Resultados y figuras |

El criterio que separa este notebook del de machine learning es si la transformación aprende algo de los datos. Las transformaciones stateless, que tratan a cada cliente mirando solo sus propios valores, se aplican aquí una sola vez. Las stateful, como imputar con una mediana, codificar, escalar o el PCA, se ajustan dentro del pipeline, solo con train. La sección 6.1 desarrolla el argumento completo y explica por qué así no hay fuga de información del test.

Cada celda de código va precedida por un texto que dice qué hace, por qué y qué esperamos ver. Después de las salidas que lo requieren sigue una interpretación breve. Las decisiones de este notebook se numeran M1 a M6 para distinguirlas de las 18 del EDA, que citamos por número.

| Sección | Qué hacemos | Por qué |
|---|---|---|
| 2. Configuración | Importamos las librerías y fijamos la semilla, las rutas relativas, los grupos de columnas y la función `verificar` | Que un error en otra computadora aparezca en el paso que lo causa |
| 3. Carga y validación (M1) | Leemos el CSV y controlamos columnas, ID, objetivo y `TotalCharges` | Las reglas posteriores dependen de esos supuestos |
| 4. Formato (M2) | Aplicamos las decisiones 1, 2 y 3 del EDA con una función y controlamos dominio y rangos | Son stateless y tienen que aplicarse siempre igual |
| 5. Partición (M3) | Leemos la partición del EDA (decisión 4), la controlamos y la unimos como columna | El test tiene que ser el mismo en los tres notebooks |
| 6. Reglas R1 a R9 (M4) | Reconstruimos faltantes por deducción (decisión 7 del EDA) y probamos que las reglas son stateless | Recuperan valores reales sin filtrar información del test |
| 7. Controles del resultado (M5) | Revisamos coherencia, rangos y los faltantes que quedan | Lo que queda pasa al imputador del pipeline y tiene que admitir una imputación coherente |
| 8. Guardado (M6) | Guardamos el CSV, lo releemos y registramos su huella | Que el notebook de machine learning parta de datos verificables |
| 9. Registro y ejecución | Tablas de decisiones, de lo que hereda el notebook de machine learning y de cómo correrlo | Trazabilidad entre los tres notebooks |
| 10. Referencias | Bibliografía citada | Respaldo de las justificaciones |

# 2. Configuración

Antes de tocar los datos fijamos lo que el resto del notebook da por sentado: las librerías, la semilla, las rutas de los archivos, los grupos de columnas con sus valores posibles y una función de control. Lo separamos en cuatro celdas para poder revisar cada parte por su cuenta.

## 2.1 Librerías

Usamos `pandas` para las tablas y `numpy` para elegir valores fila por fila con `np.where` en las reglas de la sección 6. De scikit-learn solo necesitamos `train_test_split`, para comparar la partición guardada con la que da el sorteo en esta computadora y para recrearla si faltara el archivo. `os` arma las rutas y comprueba que los archivos existan, y `hashlib` calcula la huella del CSV de salida.

Esperamos ver la versión de Python y la de cada librería. Si en otra computadora un resultado difiere, lo primero es compararlas con `requirements.txt`.

In [1]:
import os                                    # rutas relativas y existencia de archivos
import hashlib                               # huella digital del archivo que guardamos
import platform                              # versión de Python

import numpy as np                           # cálculo numérico (np.where en las reglas)
import pandas as pd                          # tablas de datos (DataFrames)
import sklearn                               # scikit-learn (se importa para ver su versión)
from sklearn.model_selection import train_test_split   # para controlar o recrear la partición

print('Python      ', platform.python_version())   # versión de Python
for lib in [pd, np, sklearn]:                        # recorro las librerías importadas
    print(f'{lib.__name__:12s}', lib.__version__)    # nombre alineado y versión

Python       3.14.6
pandas       3.0.3
numpy        2.4.6
sklearn      1.9.0


Las versiones coinciden con las fijadas en `requirements.txt`: pandas 3.0.3, numpy 2.4.6 y scikit-learn 1.9.0, con Python 3.14.

## 2.2 Semilla y rutas

La semilla fija el punto de partida del generador de números aleatorios. Con la misma semilla, una función que sortea, como `train_test_split` o `df.sample`, elige siempre los mismos elementos y el resultado se puede reproducir. Usamos `SEED = 42` en los tres notebooks porque con esa semilla el EDA definió la partición train/test (decisión 4), y cualquier control contra ese sorteo tiene que repetirla. Aquí la semilla interviene dos veces: al recalcular la partición como referencia (sección 5) y al elegir los 300 clientes de la prueba de la sección 6.

Las rutas son relativas a la carpeta del notebook, de modo que funcionan en cualquier computadora que tenga la carpeta `data/` al lado. Son tres archivos: el CSV original, la partición que guardó el EDA y el dataset que deja este notebook. Leemos la partición guardada en vez de volver a sortearla porque el sorteo depende de la implementación interna de scikit-learn. Con otra versión, la misma semilla podría mandar a otros clientes al test, y el archivo evita ese riesgo porque fija la lista de clientes de cada conjunto. `HUELLA_REGISTRADA` guarda el comienzo de la huella SHA-256 del CSV de salida, que comparamos en la sección 8.

Esperamos ver la configuración impresa y que los dos archivos de entrada existan.

In [2]:
SEED = 42                                                          # misma semilla que el EDA y el notebook de machine learning
CARPETA_DATOS = 'data'                                             # carpeta de datos, al lado del notebook
RUTA_CRUDO = os.path.join(CARPETA_DATOS, 'telco_churn_clusterai.csv')       # CSV original
RUTA_PARTICION = os.path.join(CARPETA_DATOS, 'particion_train_test.csv')    # partición que guardó el EDA
RUTA_SALIDA = os.path.join(CARPETA_DATOS, 'telco_manipulado.csv')           # dataset que deja este notebook
HUELLA_REGISTRADA = 'b61cf37c562dfe9e'                             # huella del CSV de salida al preparar el trabajo

print('Semilla:', SEED)                                            # semilla de los tres notebooks
for nombre, ruta in [('CSV original', RUTA_CRUDO), ('Partición del EDA', RUTA_PARTICION)]:   # los dos archivos de entrada
    print(f'{nombre:18s} {ruta:32s} existe: {os.path.exists(ruta)}')   # ruta y si el archivo está
print(f'{"Salida":18s} {RUTA_SALIDA:32s} (la escribe este notebook)')   # archivo que se genera en la sección 8
print('Huella registrada de la salida:', HUELLA_REGISTRADA)        # se compara en la sección 8

Semilla: 42
CSV original       data/telco_churn_clusterai.csv   existe: True
Partición del EDA  data/particion_train_test.csv    existe: True
Salida             data/telco_manipulado.csv        (la escribe este notebook)
Huella registrada de la salida: b61cf37c562dfe9e


## 2.3 Columnas y valores posibles

Agrupamos según su papel 21 de las 22 columnas del CSV. La que queda afuera es la primera, `Unnamed: 0`: es una copia del índice y se elimina en la sección 4.

- `customerID` es solo un identificador. Cada cliente tiene un código distinto, de modo que la columna no contiene ningún patrón que se repita entre clientes y el modelo no tiene nada que aprender de ella. La conservamos para unir la partición y para rastrear cada fila hasta el CSV original, pero nunca entra al modelo.
- `Churn` es la variable objetivo: indica si el cliente se dio de baja.
- Las otras 19 son las variables explicativas: 3 numéricas y 16 categóricas. A las categóricas las dividimos en datos del cliente, servicios contratados y condiciones de la cuenta.
- `ADICIONALES` reúne los seis servicios que solo se pueden contratar con internet. Las reglas R4 y R5 los usan juntos.

`DOMINIO` lista los valores posibles de cada categórica y del objetivo, los mismos que relevó el EDA. En la sección 4 lo usamos para detectar errores de tipeo.

Esperamos ver 3 numéricas, 16 categóricas y 17 variables con dominio definido: las 16 categóricas y `Churn`.

In [3]:
COL_ID = 'customerID'                                              # identificador del cliente
TARGET = 'Churn'                                                   # variable objetivo
COLS_NUM = ['tenure', 'MonthlyCharges', 'TotalCharges']            # numéricas
COLS_DEMOGRAFICAS = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']   # datos del cliente
COLS_SERVICIOS = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',   # telefonía, internet y los primeros adicionales
                  'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']   # servicios
COLS_CUENTA = ['Contract', 'PaperlessBilling', 'PaymentMethod']   # condiciones de la cuenta
COLS_CAT = COLS_DEMOGRAFICAS + COLS_SERVICIOS + COLS_CUENTA       # las 16 categóricas
ADICIONALES = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',   # los seis adicionales (sigue abajo)
               'TechSupport', 'StreamingTV', 'StreamingMovies']    # servicios que requieren internet

SI_NO = ['No', 'Yes']                                              # valores de las binarias
DOMINIO = {col: SI_NO for col in ['SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',   # diccionario columna -> valores posibles
                                  'PaperlessBilling', TARGET]}     # binarias No/Yes
DOMINIO['gender'] = ['Female', 'Male']                             # género
DOMINIO['MultipleLines'] = ['No', 'No phone service', 'Yes']       # líneas múltiples
DOMINIO['InternetService'] = ['DSL', 'Fiber optic', 'No']          # tipo de internet
DOMINIO.update({col: ['No', 'No internet service', 'Yes'] for col in ADICIONALES})   # los 6 adicionales
DOMINIO['Contract'] = ['Month-to-month', 'One year', 'Two year']   # contrato
DOMINIO['PaymentMethod'] = ['Bank transfer (automatic)', 'Credit card (automatic)',   # los cuatro medios de pago (sigue abajo)
                            'Electronic check', 'Mailed check']    # medio de pago

print('Identificador:', COL_ID, '| objetivo:', TARGET)             # columnas que no son variables explicativas
print(f'Numéricas ({len(COLS_NUM)}):', COLS_NUM)                    # las 3 numéricas
print(f'Categóricas ({len(COLS_CAT)}): {len(COLS_DEMOGRAFICAS)} del cliente, '   # total de categóricas
      f'{len(COLS_SERVICIOS)} de servicios y {len(COLS_CUENTA)} de la cuenta')   # cómo se reparten las 16
print(f'Servicios adicionales ({len(ADICIONALES)}):', ADICIONALES)  # los que requieren internet
print('Variables con dominio definido:', len(DOMINIO))             # 16 categóricas y el objetivo

Identificador: customerID | objetivo: Churn
Numéricas (3): ['tenure', 'MonthlyCharges', 'TotalCharges']
Categóricas (16): 4 del cliente, 9 de servicios y 3 de la cuenta
Servicios adicionales (6): ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
Variables con dominio definido: 17


## 2.4 Función de control

`verificar` recibe una condición y un mensaje. Si la condición se cumple, imprime `OK` y el mensaje; si no, lanza un error que detiene el notebook con el nombre del control que falló. La usamos con cada supuesto del que dependen los pasos siguientes. Así, si el notebook se ejecuta con otro archivo o desde otra carpeta, el error aparece en el paso que lo causa y no varias celdas después, en el modelo.

El primer control comprueba que el CSV original esté donde lo buscamos. Esperamos un `OK`.

In [4]:
def verificar(condicion, mensaje):   # control con mensaje claro
    # si la condición no se cumple, detiene el notebook diciendo qué control falló
    if not condicion:                                              # el control falló
        raise ValueError('Falló el control: ' + mensaje)           # corto la ejecución acá
    print('OK -', mensaje)                                         # el control pasó


verificar(os.path.exists(RUTA_CRUDO), f'existe {RUTA_CRUDO} (el notebook corre desde tp_final_telco/)')   # ¿encuentro el CSV?

OK - existe data/telco_churn_clusterai.csv (el notebook corre desde tp_final_telco/)


# 3. Carga del CSV y validación de su estructura

## 3.1 Lectura

Leemos el CSV original en `df_crudo` y no lo modificamos en ningún paso. Las transformaciones trabajan sobre copias, así que siempre podemos comparar contra los datos de partida. Esperamos 7.043 filas y 22 columnas, y que en las primeras filas ya aparezcan faltantes (`NaN`).

In [5]:
df_crudo = pd.read_csv(RUTA_CRUDO)                                 # leo el CSV sin tocar nada
print(f'Shape: {df_crudo.shape[0]} filas x {df_crudo.shape[1]} columnas')   # tamaño
df_crudo.head(3)                                                   # primeras filas

Shape: 7043 filas x 22 columnas


,Unnamed: 0,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,7590-VHVEG,Female,NaN,Yes,No,1.0,No,No phone service,DSL,...,No,No,No,No,Month-to-month,NaN,Electronic check,29.85,29.85,No
1,1,5575-GNVDE,Male,0.0,No,No,34.0,Yes,No,NaN,...,Yes,No,No,No,One year,No,Mailed check,NaN,1889.5,No
2,2,3668-QPYBK,Male,0.0,No,No,2.0,NaN,No,NaN,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


La tabla tiene el tamaño esperado. Ya en las tres primeras filas faltan valores de `SeniorCitizen`, `PhoneService`, `InternetService`, `PaperlessBilling` y `MonthlyCharges`. `SeniorCitizen` aparece como `0.0` y `tenure` como `1.0` porque pandas guarda como decimales las columnas numéricas con faltantes: `NaN` es un valor de punto flotante.

## 3.2 Columnas e identificador

Antes de transformar comprobamos cuatro supuestos:

- que estén exactamente las 22 columnas que analizó el EDA;
- que `customerID` no se repita, es decir, que haya una fila por cliente;
- que el identificador y el objetivo estén completos;
- que `Unnamed: 0` coincida con el número de fila.

Si un `customerID` se repitiera, la unión con la partición de la sección 5 podría asignarle dos conjuntos al mismo cliente. Una fila sin objetivo no serviría para entrenar ni para evaluar. El último control respalda la decisión 1 del EDA, que elimina `Unnamed: 0`. Esperamos cuatro `OK`.

In [6]:
columnas_esperadas = {'Unnamed: 0', COL_ID, TARGET} | set(COLS_CAT) | set(COLS_NUM)   # las 22 del CSV
verificar(set(df_crudo.columns) == columnas_esperadas, 'el CSV tiene exactamente las 22 columnas esperadas')   # estructura
verificar(df_crudo[COL_ID].is_unique, 'customerID es único (un cliente por fila)')   # sin clientes repetidos
verificar(df_crudo[COL_ID].notna().all() and df_crudo[TARGET].notna().all(),   # ¿ID y objetivo sin NaN?
          'customerID y Churn no tienen faltantes')                # ID y objetivo completos
verificar((df_crudo['Unnamed: 0'] == df_crudo.index).all(),        # ¿Unnamed: 0 repite el número de fila?
          'Unnamed: 0 es una copia del índice de filas')           # confirma la decisión 1 del EDA

OK - el CSV tiene exactamente las 22 columnas esperadas
OK - customerID es único (un cliente por fila)
OK - customerID y Churn no tienen faltantes
OK - Unnamed: 0 es una copia del índice de filas


## 3.3 Contenido de `TotalCharges`

`TotalCharges` se lee como texto porque algunas celdas tienen un espacio en blanco en lugar de un número. `pd.to_numeric(..., errors='coerce')` convierte cada valor a número y deja `NaN` donde no puede. Comparando el resultado con la columna original aislamos los textos que no se pudieron convertir. La decisión 2 del EDA supone que todos son blancos. Si apareciera otro texto, por ejemplo un número con coma decimal, convertirlo a `NaN` borraría un dato real. Esperamos 11 valores no numéricos, todos en blanco.

In [7]:
print('Tipo de TotalCharges en el CSV:', df_crudo['TotalCharges'].dtype)   # se lee como texto
# TotalCharges: lo que no se puede convertir a número tiene que ser solo espacios en blanco
tc_numerico = pd.to_numeric(df_crudo['TotalCharges'], errors='coerce')   # intento convertir
no_numericos = df_crudo.loc[tc_numerico.isna() & df_crudo['TotalCharges'].notna(), 'TotalCharges']   # texto no convertible
print('Textos distintos que no son números:', no_numericos.unique().tolist())   # qué contienen esas celdas
verificar(no_numericos.str.strip().eq('').all(),                   # ¿todos son blancos?
          f'los {len(no_numericos)} valores no numéricos de TotalCharges son espacios en blanco')   # solo blancos

Tipo de TotalCharges en el CSV: str
Textos distintos que no son números: [' ']
OK - los 11 valores no numéricos de TotalCharges son espacios en blanco


El archivo tiene 7.043 filas y la estructura que analizó el EDA. Los 11 valores no numéricos de `TotalCharges` son un espacio en blanco, que la decisión 2 del EDA atribuye a clientes sin facturas todavía.

> **Decisión M1:** validar la estructura del CSV antes de transformarlo y detener la ejecución si algo no coincide con lo que analizó el EDA.
>
> **Justificación:** las reglas de este notebook se apoyan en supuestos sobre los datos, por ejemplo que los únicos textos de `TotalCharges` sean blancos. Si un supuesto falla y el código sigue, el error aparece recién en el modelo, lejos de su causa, o no aparece. Validar la estructura es el primer paso del flujo de un proyecto de aprendizaje automático (Géron, 2022, cap. 2). Aquí además cumple un fin práctico: si el notebook se corre con otra versión del archivo o desde otra carpeta, el mensaje indica qué cambió.

# 4. Corrección de formato (decisiones 1, 2 y 3 del EDA)

## 4.1 La función `corregir_formato`

Reunimos en una función las tres correcciones de formato del EDA:

- **Decisión 1.** Eliminar `Unnamed: 0`, que repite el número de fila y no aporta información.
- **Decisión 2.** Convertir `TotalCharges` a número. Los 11 blancos quedan como `NaN` y la regla R1 los completa en la sección 6.
- **Decisión 3.** Pasar `SeniorCitizen` de 0/1 a No/Yes, el formato de las otras binarias, para que el pipeline la codifique igual que a ellas.

Las tres son stateless: cada valor se transforma mirando solo su propia celda, así que da lo mismo aplicarlas antes o después de separar train y test (el argumento completo está en la sección 6.1). Escribirlas como función asegura que se apliquen siempre igual y en el mismo orden, también a un cliente nuevo (Géron, 2022, cap. 2).

Esperamos 21 columnas, las tres numéricas como `float64` y `SeniorCitizen` como texto. También imprimimos los faltantes de las dos columnas convertidas, antes y después.

In [8]:
def corregir_formato(crudo):   # decisiones 1, 2 y 3 del EDA
    # devuelve una copia con las correcciones de formato; no modifica el original
    d = crudo.drop(columns=['Unnamed: 0']).copy()                  # decisión 1: saco la copia del índice
    d['TotalCharges'] = pd.to_numeric(d['TotalCharges'], errors='coerce')   # decisión 2: texto a número (blancos -> NaN)
    d['SeniorCitizen'] = d['SeniorCitizen'].map({0: 'No', 1: 'Yes'})        # decisión 3: 0/1 -> No/Yes (NaN queda NaN)
    return d                                                       # datos con el formato corregido


df = corregir_formato(df_crudo)                                    # aplico las correcciones
print('Shape:', df.shape)                                          # 21 columnas
print(df[COLS_NUM + ['SeniorCitizen']].dtypes.to_string())         # tipos resultantes
print('NaN en TotalCharges:', df_crudo['TotalCharges'].isna().sum(), '->', df['TotalCharges'].isna().sum())     # los blancos pasan a NaN
print('NaN en SeniorCitizen:', df_crudo['SeniorCitizen'].isna().sum(), '->', df['SeniorCitizen'].isna().sum())  # el map conserva los faltantes
print(df['SeniorCitizen'].value_counts(dropna=False).to_string())  # valores de SeniorCitizen después de traducir

Shape: (7043, 21)
tenure            float64
MonthlyCharges    float64
TotalCharges      float64
SeniorCitizen         str
NaN en TotalCharges: 0 -> 11
NaN en SeniorCitizen: 916 -> 916
SeniorCitizen
No     5137
Yes     990
NaN     916


Quedan 21 columnas, con las tres numéricas como `float64` y `SeniorCitizen` como texto. `TotalCharges` pasa de 0 a 11 faltantes, que son los blancos, y `SeniorCitizen` conserva sus 916.

## 4.2 Faltantes creados por la conversión

Un `map` deja como `NaN`, sin avisar, cualquier valor que no figure en su diccionario. Si el CSV tuviera un 2 en `SeniorCitizen`, o si una clave del diccionario estuviera mal escrita, aparecerían faltantes nuevos que después el imputador taparía. Controlamos que los `NaN` de `SeniorCitizen` estén en las mismas filas que en el CSV y que los de `TotalCharges` sean los originales más los blancos. Esperamos dos `OK`.

In [9]:
# los NaN de SeniorCitizen tienen que ser los mismos que en el CSV (el map no puede crear faltantes nuevos)
verificar(df['SeniorCitizen'].isna().equals(df_crudo['SeniorCitizen'].isna()),   # mismas filas con NaN antes y después
          'convertir SeniorCitizen no creó faltantes nuevos')      # solo 0 y 1 se tradujeron
verificar(df['TotalCharges'].isna().sum() == df_crudo['TotalCharges'].isna().sum() + len(no_numericos),   # NaN originales + blancos
          'los NaN de TotalCharges son los originales más los blancos')   # nada más se perdió

OK - convertir SeniorCitizen no creó faltantes nuevos
OK - los NaN de TotalCharges son los originales más los blancos


## 4.3 Valores posibles y rangos

Recorremos `DOMINIO` y comprobamos que cada categórica y el objetivo tengan solo los valores esperados. Una categoría mal escrita ("yes" por "Yes") generaría una columna aparte en la codificación one-hot del notebook de machine learning. Si apareciera solo en test, `handle_unknown='ignore'` la codificaría con ceros en todas las columnas de esa variable y el error pasaría inadvertido.

En las numéricas controlamos que `tenure` sea un número entero de meses entre 0 y 72, el rango que relevó el EDA, y que no haya montos negativos. Esperamos 17 `OK` de dominio y 2 de rangos.

In [10]:
# dominio: cada categórica solo puede tener los valores esperados (o NaN)
for col, valores in DOMINIO.items():                               # recorro las categóricas y el objetivo
    fuera = set(df[col].dropna().unique()) - set(valores)          # valores que no deberían estar
    verificar(not fuera, f'{col}: solo valores esperados')         # sin errores de tipeo

# numéricas: rangos posibles
tenure_obs = df['tenure'].dropna()                                 # antigüedades registradas
verificar((tenure_obs % 1 == 0).all() and tenure_obs.between(0, 72).all(),   # entera y en [0, 72]
          'tenure es entera y está entre 0 y 72 meses')            # antigüedad válida
verificar((df[['MonthlyCharges', 'TotalCharges']].min() >= 0).all(), 'no hay montos negativos')   # el mínimo ignora los NaN

OK - SeniorCitizen: solo valores esperados
OK - Partner: solo valores esperados
OK - Dependents: solo valores esperados
OK - PhoneService: solo valores esperados
OK - PaperlessBilling: solo valores esperados
OK - Churn: solo valores esperados
OK - gender: solo valores esperados
OK - MultipleLines: solo valores esperados
OK - InternetService: solo valores esperados
OK - OnlineSecurity: solo valores esperados
OK - OnlineBackup: solo valores esperados
OK - DeviceProtection: solo valores esperados
OK - TechSupport: solo valores esperados
OK - StreamingTV: solo valores esperados
OK - StreamingMovies: solo valores esperados
OK - Contract: solo valores esperados
OK - PaymentMethod: solo valores esperados
OK - tenure es entera y está entre 0 y 72 meses
OK - no hay montos negativos


Las 16 categóricas y el objetivo tienen solo los valores que relevó el EDA, y las numéricas están en rangos posibles. La corrección de formato no creó faltantes nuevos.

> **Decisión M2:** aplicar las decisiones 1, 2 y 3 del EDA con la función `corregir_formato` y controlar después que no haya creado faltantes y que el dominio y los rangos sean los esperados.
>
> **Justificación:** las tres transformaciones son stateless, de modo que se aplican una sola vez a todo el dataset sin fuga (sección 6.1). Escribirlas como función garantiza que se repitan igual sobre cualquier conjunto de datos (Géron, 2022, cap. 2). Los controles cubren errores que pandas no avisa: un `map` con claves mal escritas convierte todos los valores en `NaN` sin error.

# 5. Partición train/test (decisión 4 del EDA)

El EDA separó el 20% de los clientes como test antes de mirar cualquier relación con el objetivo. Guardó el `customerID` de cada cliente con su conjunto en `data/particion_train_test.csv`. En esta sección leemos ese archivo, lo controlamos y lo agregamos a los datos como una columna.

## 5.1 Partición de referencia

Primero calculamos la partición tal como la define la decisión 4: `train_test_split` con 20% de test, estratificada por `Churn` y con `random_state=SEED`. Estratificar significa sortear dentro de cada clase por separado, de modo que train y test conserven la misma proporción de clientes que se dan de baja. Esta partición queda como referencia. En el paso 5.4 medimos si coincide con el archivo guardado, y si el archivo faltara la usamos para recrearlo. Esperamos 5.634 clientes en train y 1.409 en test.

In [11]:
# la partición que define la decisión 4, calculada con la versión de scikit-learn instalada
idx_train, idx_test = train_test_split(df.index, test_size=0.2, stratify=df[TARGET], random_state=SEED)   # 80/20 estratificado
recalculada = pd.Series('train', index=df.index)                   # todos empiezan en train
recalculada.loc[idx_test] = 'test'                                 # marco los de test
print(recalculada.value_counts().to_string())                      # tamaños de la partición de referencia

train    5634
test     1409


## 5.2 Lectura del archivo de partición

Si el archivo existe, lo leemos; es el caso normal cuando el EDA ya se ejecutó. Si falta, lo recreamos con la partición de referencia, lo guardamos y lo avisamos. En ese caso lo escribimos con fin de línea `\n`, para que el archivo sea el mismo en cualquier sistema operativo. Esperamos que se lea el archivo y que muestre los mismos tamaños que la referencia.

In [12]:
if os.path.exists(RUTA_PARTICION):                                 # caso normal: el EDA ya la guardó
    particion = pd.read_csv(RUTA_PARTICION)                        # leo la partición del EDA
    print('Partición leída de', RUTA_PARTICION)                    # aviso de dónde salió
else:                                                              # el EDA no se corrió en esta computadora
    particion = pd.DataFrame({COL_ID: df[COL_ID], 'conjunto': recalculada.values})   # la recreo
    particion.to_csv(RUTA_PARTICION, index=False, lineterminator='\n')   # y la guardo (fin de línea fijo en cualquier sistema)
    print('AVISO: no estaba', RUTA_PARTICION, '- la recreé con el criterio de la decisión 4')   # aviso
print(particion['conjunto'].value_counts().to_string())            # tamaños de cada conjunto

Partición leída de data/particion_train_test.csv
conjunto
train    5634
test     1409


## 5.3 Controles de la partición

Comprobamos que cada cliente aparezca una sola vez, que la partición tenga exactamente los mismos clientes que el CSV y que solo haya dos etiquetas, `train` y `test`. Un archivo que viniera de otra versión del dataset, o que se hubiera editado a mano, fallaría en alguno de estos controles. Esperamos tres `OK`.

In [13]:
verificar(particion[COL_ID].is_unique, 'cada cliente aparece una sola vez en la partición')   # sin duplicados
verificar(set(particion[COL_ID]) == set(df[COL_ID]), 'la partición cubre exactamente a los 7.043 clientes del CSV')   # mismos clientes
verificar(set(particion['conjunto']) == {'train', 'test'}, 'solo hay dos conjuntos: train y test')   # etiquetas válidas

OK - cada cliente aparece una sola vez en la partición
OK - la partición cubre exactamente a los 7.043 clientes del CSV
OK - solo hay dos conjuntos: train y test


## 5.4 Unión con los datos

Unimos la partición a los datos por `customerID`, y cada fila recibe la columna `conjunto`. Con `validate='one_to_one'`, si un cliente apareciera repetido de cualquiera de los dos lados, el `merge` fallaría en lugar de duplicar filas. Después medimos qué porcentaje de clientes cae en el mismo conjunto que en la partición de referencia. Esperamos 22 columnas y una coincidencia del 100%.

In [14]:
df = df.merge(particion, on=COL_ID, how='left', validate='one_to_one')   # agrego la columna conjunto
print('Shape después de unir:', df.shape)                          # una columna más: conjunto
coincide = (df['conjunto'].values == recalculada.values).mean()    # coincidencia con el cálculo actual
print(f'Coincidencia con train_test_split en esta computadora: {coincide:.1%}')   # muestro el resultado

Shape después de unir: (7043, 22)
Coincidencia con train_test_split en esta computadora: 100.0%


## 5.5 Estratificación

Calculamos el porcentaje de churn en cada conjunto. Si la estratificación funcionó, tiene que ser prácticamente el mismo en los dos. Controlamos que la diferencia sea menor a 0,1 puntos porcentuales y que el test tenga 1.409 clientes.

In [15]:
tasas = df.groupby('conjunto')[TARGET].apply(lambda s: (s == 'Yes').mean() * 100)   # % de churn por conjunto
print('Tasa de churn (%):', tasas.round(2).to_dict())             # muestro el resultado
verificar((df['conjunto'] == 'test').sum() == 1409 and abs(tasas['train'] - tasas['test']) < 0.1,   # tamaño del test y diferencia de tasas
          'test tiene 1.409 clientes y la misma tasa de churn que train (estratificación)')   # tamaños y estratos

Tasa de churn (%): {'test': 26.54, 'train': 26.54}
OK - test tiene 1.409 clientes y la misma tasa de churn que train (estratificación)


La partición tiene 5.634 clientes en train y 1.409 en test, con 26,54% de churn en ambos, y coincide en un 100% con la que calcula `train_test_split` en esta computadora. Si en otra la coincidencia fuera menor, el notebook seguiría usando el archivo, que es lo que mantiene el mismo test en los tres notebooks.

> **Decisión M3:** leer la partición desde `data/particion_train_test.csv`, verificar que cubra a cada cliente una sola vez y conserve la estratificación, y unirla como columna `conjunto`. Si el archivo falta, recrearlo con el criterio de la decisión 4 del EDA.
>
> **Justificación:** si el test cambiara entre el EDA y el modelo, algunas decisiones del EDA se habrían tomado con clientes que después se usan para evaluar. La estimación del error de generalización dejaría de ser válida (James et al., 2023, sec. 5.1; Géron, 2022, cap. 2). Guardar la partición la vuelve independiente de la versión de scikit-learn. Unirla como columna, con `validate='one_to_one'`, hace fallar el `merge` si un cliente apareciera repetido de cualquiera de los dos lados. Además deja a cada fila con su etiqueta, de modo que el notebook de machine learning separa train y test con un filtro (VanderPlas, 2016, cap. 3).

# 6. Reconstrucción de faltantes con las reglas R1 a R9 (decisión 7 del EDA)

El CSV tiene faltantes en 14 variables, entre el 13% y el 17% de cada una, y solo el 12% de los clientes tiene la fila completa (lo medimos en la sección 7). El EDA no encontró evidencia en contra de que esos faltantes sean MCAR (decisión 6), es decir, que la probabilidad de que falte un dato no dependa de su valor ni del resto de la fila. También mostró que muchos se pueden deducir de otras columnas del mismo cliente, porque entre ellas hay relaciones que se cumplen sin excepciones (decisión 7). Deducir un valor recupera el dato real del cliente, mientras que una mediana o una moda le asignan un valor típico del conjunto.

## 6.1 Las reglas y por qué valen

| Regla | Si falta | y se observa | se completa con | Tipo |
|---|---|---|---|---|
| R1 | `TotalCharges` (blanco) y `tenure` | blanco en `TotalCharges` (cliente sin facturas) | `TotalCharges = 0`, `tenure = 0` | exacta |
| R2 | `PhoneService` | `MultipleLines` | "No" si es "No phone service"; si no, "Yes" | exacta |
| R3 | `MultipleLines` | `PhoneService = No` | "No phone service" | exacta |
| R4 | `InternetService` | algún adicional = "No internet service" | "No" | exacta |
| R5 | un adicional | `InternetService = No` | "No internet service" | exacta |
| R6 | `MonthlyCharges` | `tenure > 0` y `TotalCharges` | `TotalCharges / tenure` | aproximada |
| R7 | `tenure` | `MonthlyCharges` y `TotalCharges` | `round(TotalCharges / MonthlyCharges)`, en [1, 72] | aproximada |
| R8 | `InternetService` | `PhoneService = No` | "DSL" | empírica, sin excepciones en train (425/425) |
| R9 | `PhoneService` | `InternetService` = fibra o sin internet | "Yes" | empírica, sin excepciones en train (425/425) |

La lógica de negocio detrás de cada regla:

- **R1.** `TotalCharges` es la suma de lo facturado al cliente. Un blanco corresponde a un cliente que todavía no recibió facturas (decisión 2), así que su total es 0 y su antigüedad, 0 meses.
- **R2 y R3.** `MultipleLines` vale "No phone service" exactamente cuando el cliente no tiene teléfono. Si `MultipleLines` dice "No phone service", `PhoneService` es "No"; si dice "No" o "Yes", el cliente tiene línea y `PhoneService` es "Yes" (R2). En sentido inverso, un cliente sin teléfono no puede tener líneas múltiples (R3).
- **R4 y R5.** Los seis servicios adicionales (seguridad, backup, protección del equipo, soporte técnico y streaming de TV y de películas) funcionan sobre la conexión a internet. Un cliente sin internet tiene los seis en "No internet service" (R5). Basta que uno solo de ellos diga "No internet service" para saber que el cliente no tiene internet (R4).
- **R6 y R7.** Si el cargo mensual no cambió, el total facturado es la antigüedad por el cargo: `TotalCharges ≈ tenure × MonthlyCharges`. Despejando se obtiene el cargo (R6) o la antigüedad (R7). Son aproximadas porque el precio pudo cambiar con el tiempo. El EDA las validó en train con 4.056 clientes que tenían las tres numéricas: el error absoluto medio de R6 es 1,74 en el cargo contra 25,90 de la mediana, y el de R7 es 0,68 meses contra 21,86. La mediana es una referencia simple, que no usa ninguna otra columna. Sirve para mostrar que las reglas aprovechan la relación entre variables, aunque no las compara con imputadores más elaborados. R7 redondea a meses enteros y acota el resultado a [1, 72], porque un cliente con facturas tiene al menos un mes y 72 es la antigüedad máxima.
- **R8 y R9.** Se apoyan en una regularidad de los datos, que atribuimos a la oferta comercial de la empresa: todos los clientes sin teléfono tienen DSL. Entonces un cliente sin teléfono que no informa su internet tiene DSL (R8). Leída al revés, la misma relación dice que un cliente con fibra o sin internet tiene teléfono (R9). El EDA la observó con el dataset completo (su sección 4.2) y después la confirmó usando solo train: los 425 clientes de train sin teléfono tienen DSL.

**Por qué aplicar las reglas aquí no filtra información del test.** Una transformación stateless trata a cada cliente mirando solo sus propios valores, como convertir un texto a número o deducir un dato a partir de otras columnas de la misma fila. No calcula estadísticas del conjunto, como medias, modas o conteos. Por eso da el mismo resultado en train, en test o con un cliente que llegue después. Un cliente de test recibe el mismo valor que recibiría si fuera el único cliente del dataset. Las nueve reglas, igual que las correcciones de formato de la sección 4, cumplen esa condición y se pueden aplicar una sola vez a todo el dataset. Las secciones 6.4 y 6.5 lo comprueban.

Una transformación stateful, como imputar con una mediana, codificar, escalar o el PCA, calcula parámetros con los datos que recibe. Si se ajustara con todos los clientes, el test influiría en el entrenamiento y la evaluación final sería optimista. Por eso esas transformaciones se ajustan dentro del pipeline, solo con train en cada fold (Géron, 2022, cap. 2).

Las relaciones en que se apoyan las reglas tienen tres orígenes. R1 a R5 salen de la definición de las variables. R6 y R7 son aproximaciones que el EDA validó con datos de train. R8 y R9 son una regularidad empírica: se observaron con el dataset completo y se confirmaron en train sin excepciones.

## 6.2 Implementación

La función `reconstruir_faltantes` aplica las reglas en orden sobre una copia de los datos. Cada regla sigue el mismo patrón. Una máscara booleana `m` marca las filas donde falta el valor y se observa la condición, `d.loc[m, columna]` completa solo esas filas y el registro anota cuántas fueron. Las filas que no cumplen la condición conservan su `NaN` y pasan al imputador del pipeline.

El orden importa en tres puntos. R1 va primero porque fija `tenure = 0` en los clientes nuevos; si no, R7 dividiría un total de 0 por el cargo y les asignaría una antigüedad de 1 mes. R8 usa el `PhoneService` que pudo completar R2, y R9 el `InternetService` que pudo completar R4. Es la misma función que usa el EDA. Esta celda solo la define, por lo que no produce salida.

In [16]:
def reconstruir_faltantes(datos):   # reglas R1 a R9 (decisión 7 del EDA)
    # aplica las reglas R1 a R9 fila por fila; devuelve los datos completados y cuántos valores llenó cada regla
    d = datos.copy()                                           # trabajo sobre una copia
    registro = {}                                              # cuántos valores completa cada regla

    # R1: TotalCharges en blanco (NaN después de convertir) = cliente nuevo -> TotalCharges 0 y tenure 0
    blanco = d['TotalCharges'].isna()                          # filas con TotalCharges en blanco
    m = blanco & d['tenure'].isna()                            # de esas, las que además no tienen tenure
    d.loc[m, 'tenure'] = 0                                     # sin facturas, la antigüedad es 0
    registro['R1 tenure'] = int(m.sum())                       # anoto cuántas completé
    d.loc[blanco, 'TotalCharges'] = 0.0                        # sin facturas, el total es 0
    registro['R1 TotalCharges'] = int(blanco.sum())            # anoto cuántas completé

    # R2: PhoneService desde MultipleLines
    m = d['PhoneService'].isna() & d['MultipleLines'].notna()  # falta PhoneService pero sé MultipleLines
    d.loc[m, 'PhoneService'] = np.where(d.loc[m, 'MultipleLines'] == 'No phone service', 'No', 'Yes')   # No o Yes según MultipleLines
    registro['R2 PhoneService'] = int(m.sum())                 # anoto cuántas completé

    # R3: MultipleLines desde PhoneService = No
    m = d['MultipleLines'].isna() & (d['PhoneService'] == 'No')   # sin teléfono -> "No phone service"
    d.loc[m, 'MultipleLines'] = 'No phone service'             # completo
    registro['R3 MultipleLines'] = int(m.sum())                # anoto cuántas completé

    # R4: InternetService = No si algún adicional dice "No internet service"
    m = d['InternetService'].isna() & (d[ADICIONALES] == 'No internet service').any(axis=1)   # falta y algún adicional lo delata
    d.loc[m, 'InternetService'] = 'No'                         # completo
    registro['R4 InternetService'] = int(m.sum())              # anoto cuántas completé

    # R5: cada adicional = "No internet service" si el cliente no tiene internet
    for col in ADICIONALES:                                    # recorro los 6 adicionales
        m = d[col].isna() & (d['InternetService'] == 'No')     # falta el adicional y no tiene internet
        d.loc[m, col] = 'No internet service'                  # completo
        registro[f'R5 {col}'] = int(m.sum())                   # anoto cuántas completé

    # R6: MonthlyCharges = TotalCharges / tenure (redondeado a centavos)
    m = d['MonthlyCharges'].isna() & (d['tenure'] > 0) & d['TotalCharges'].notna()   # falta el cargo y puedo dividir
    d.loc[m, 'MonthlyCharges'] = (d.loc[m, 'TotalCharges'] / d.loc[m, 'tenure']).round(2)   # cargo = total / meses
    registro['R6 MonthlyCharges'] = int(m.sum())               # anoto cuántas completé

    # R7: tenure = round(TotalCharges / MonthlyCharges), acotado al rango válido [1, 72]
    m = d['tenure'].isna() & d['MonthlyCharges'].notna() & d['TotalCharges'].notna()   # falta la antigüedad y puedo dividir
    d.loc[m, 'tenure'] = (d.loc[m, 'TotalCharges'] / d.loc[m, 'MonthlyCharges']).round().clip(1, 72)   # meses = total / cargo
    registro['R7 tenure'] = int(m.sum())                       # anoto cuántas completé

    # R8: sin teléfono -> el internet es DSL
    m = d['InternetService'].isna() & (d['PhoneService'] == 'No')   # falta internet y no tiene teléfono
    d.loc[m, 'InternetService'] = 'DSL'                        # completo
    registro['R8 InternetService'] = int(m.sum())              # anoto cuántas completé

    # R9: con fibra o sin internet -> tiene teléfono
    m = d['PhoneService'].isna() & d['InternetService'].isin(['Fiber optic', 'No'])   # falta teléfono y el internet lo implica
    d.loc[m, 'PhoneService'] = 'Yes'                           # completo
    registro['R9 PhoneService'] = int(m.sum())                 # anoto cuántas completé

    return d, registro                                         # datos completados y el registro

## 6.3 Aplicación a todo el dataset

Aplicamos las reglas a todos los clientes y mostramos cuántos valores completó cada una. También imprimimos el total de faltantes en las 19 variables antes y después de las reglas. El total tiene que coincidir con los 3.398 que obtuvo el EDA. Como un total igual podría esconder dos reglas que se compensan, comparamos además cada regla con la cantidad que imprimió el EDA en su sección 6.4, copiada en `REGISTRO_EDA`. Esperamos dos `OK`.

In [17]:
df_rec, registro = reconstruir_faltantes(df)                       # aplico las reglas a todo el dataset
tabla_registro = pd.Series(registro, name='valores completados').to_frame()   # registro como tabla
tabla_registro.loc['TOTAL'] = tabla_registro['valores completados'].sum()     # fila de total
print(tabla_registro.to_string())                                  # muestro la tabla
print('Faltantes en las 19 variables:', df[COLS_CAT + COLS_NUM].isna().sum().sum(),   # NaN antes de las reglas
      '->', df_rec[COLS_CAT + COLS_NUM].isna().sum().sum())         # antes y después de las reglas
verificar(tabla_registro.loc['TOTAL', 'valores completados'] == 3398,   # comparo el total con el del EDA
          'las reglas recuperan los mismos 3.398 valores que en el EDA')   # mismo resultado que el EDA
REGISTRO_EDA = {'R1 tenure': 1, 'R1 TotalCharges': 11, 'R2 PhoneService': 788, 'R3 MultipleLines': 75,   # salida del EDA, sección 6.4
                'R4 InternetService': 183, 'R5 OnlineSecurity': 194, 'R5 OnlineBackup': 178,   # (el EDA imprime solo las reglas
                'R6 MonthlyCharges': 1039, 'R7 tenure': 758, 'R8 InternetService': 97, 'R9 PhoneService': 74}   # que completan algo)
verificar({k: v for k, v in registro.items() if v > 0} == REGISTRO_EDA,   # comparo regla por regla, sin modificar nada
          'cada regla recupera la misma cantidad de valores que en el EDA')   # descarta reglas que se compensen

                     valores completados
R1 tenure                              1
R1 TotalCharges                       11
R2 PhoneService                      788
R3 MultipleLines                      75
R4 InternetService                   183
R5 OnlineSecurity                    194
R5 OnlineBackup                      178
R5 DeviceProtection                    0
R5 TechSupport                         0
R5 StreamingTV                         0
R5 StreamingMovies                     0
R6 MonthlyCharges                   1039
R7 tenure                            758
R8 InternetService                    97
R9 PhoneService                       74
TOTAL                               3398
Faltantes en las 19 variables: 13959 -> 10561
OK - las reglas recuperan los mismos 3.398 valores que en el EDA
OK - cada regla recupera la misma cantidad de valores que en el EDA


Las reglas recuperan 3.398 valores, el mismo total que en el EDA y la misma cantidad en cada regla, y los faltantes de las 19 variables bajan de 13.959 a 10.561. Las que más aportan son R6 (1.039 cargos mensuales), R2 (788 datos de teléfono) y R7 (758 antigüedades). R1 completa los 11 blancos de `TotalCharges` y la antigüedad de uno de esos clientes, que también faltaba. R5 completa 372 adicionales; en cuatro de los seis no completa nada porque esas columnas no tienen faltantes en el CSV.

## 6.4 Prueba 1: train y test por separado

En la sección 6.1 argumentamos que las reglas no usan otras filas. Lo comprobamos con dos pruebas. La primera aplica la función a train y a test por separado, junta los resultados y los compara con los de todo el dataset. Si una regla usara un estadístico del conjunto que difiriera entre train y test, el resultado cambiaría. Es una condición necesaria pero no suficiente, porque un estadístico que coincidiera en los dos conjuntos (una moda, por ejemplo) pasaría la prueba. Esperamos un `OK`.

In [18]:
# prueba 1: por conjunto
partes = [reconstruir_faltantes(df[df['conjunto'] == c])[0] for c in ['train', 'test']]   # aplico a cada conjunto por separado
print('Filas procesadas por separado (train, test):', [len(p) for p in partes])   # tamaño de cada parte
por_separado = pd.concat(partes).loc[df.index]                     # junto y reordeno como el original
verificar(por_separado.equals(df_rec),                             # ¿idéntico celda por celda?
          'aplicar las reglas a train y a test por separado da exactamente lo mismo que aplicarlas a todo')   # condición necesaria

Filas procesadas por separado (train, test): [5634, 1409]
OK - aplicar las reglas a train y a test por separado da exactamente lo mismo que aplicarlas a todo


## 6.5 Prueba 2: cliente por cliente

La segunda prueba elige 300 clientes al azar con la semilla del notebook y aplica la función a cada uno solo, sin ninguna otra fila de la que aprender. Reproduce la situación de un cliente nuevo que llega de a uno. Si el resultado coincide con el de todo el dataset, ninguna regla depende de las demás filas. Para que la prueba no sea trivial imprimimos cuántos valores completaron las reglas en esos 300 clientes. Esperamos un `OK`.

In [19]:
# prueba 2: cliente por cliente
muestra = df.sample(300, random_state=SEED)                        # 300 clientes al azar
uno_por_uno = pd.concat([reconstruir_faltantes(muestra.loc[[i]])[0] for i in muestra.index])   # cada cliente, solo
completados = muestra[COLS_CAT + COLS_NUM].isna().sum().sum() - uno_por_uno[COLS_CAT + COLS_NUM].isna().sum().sum()   # NaN que se llenaron
print(f'Clientes procesados de a uno: {len(muestra)}; valores completados en ellos: {completados}')   # la prueba tiene casos reales
verificar(uno_por_uno.equals(df_rec.loc[muestra.index]),           # ¿idéntico celda por celda?
          'aplicar las reglas a cada cliente por separado da lo mismo que aplicarlas a todo')   # stateless fila por fila

Clientes procesados de a uno: 300; valores completados en ellos: 141
OK - aplicar las reglas a cada cliente por separado da lo mismo que aplicarlas a todo


Las dos pruebas dan un resultado idéntico, celda por celda. La segunda reproduce la situación de un cliente nuevo: procesado solo, recibe exactamente lo mismo que dentro del dataset completo. En esos 300 clientes las reglas completaron 141 valores, así que la prueba abarca casos reales.

> **Decisión M4:** aplicar las reglas R1 a R9 (decisión 7 del EDA) a todo el dataset, con la misma función y en el mismo orden, y comprobar que el resultado no depende de las filas con que se aplican.
>
> **Justificación:** las reglas devuelven el valor real o uno muy cercano, mientras que un imputador pondría el mismo valor a todos los clientes; aprovechar la relación entre variables completa mejor que la media (James et al., 2023, sec. 12.3). Las dos pruebas confirman que son stateless, así que aplicarlas antes del pipeline no filtra información del test (sección 6.1). En los datos originales el EDA no encontró evidencia en contra de MCAR (decisión 6). Después de las reglas, los faltantes de servicios que quedan son MAR, porque dependen de tener el servicio (sección 6.5 del EDA y sección 7.5 de este notebook). Con MCAR y con MAR el mecanismo de pérdida se puede ignorar, porque depende solo de lo observado (Murphy, 2022, sec. 1.5.5). Por eso el cambio no invalida la imputación dentro del pipeline.

# 7. Controles del resultado

Antes de guardar revisamos tres cosas: que la reconstrucción no haya creado contradicciones entre columnas, que las numéricas sigan en rangos posibles y que los faltantes que quedan admitan la imputación que hace el pipeline (decisión 8 del EDA).

## 7.1 Coherencia entre columnas

Una regla mal escrita podría dejar a un cliente con `PhoneService = Yes` y `MultipleLines` en "No phone service", o con internet y un adicional en "No internet service". Volvemos a comprobar las relaciones de la sección 6.1 sobre los datos completados, mirando solo las filas que tienen los dos datos. Son cuatro controles: telefonía, internet, que los adicionales estén todos o ninguno en "No internet service", y la oferta comercial de R8.

El cuarto es un control de implementación. Como R8 escribe "DSL" justamente en los clientes sin teléfono, después de aplicarla se cumple en parte por construcción. Lo que agrega es que R8 se aplicó bien y que los valores observados no la contradicen. La regla empírica no se prueba aquí: el EDA la verificó con datos de train (425 de 425). Esperamos cuatro `OK`.

In [20]:
# las reglas entre columnas tienen que seguir valiendo sin excepciones
tel = df_rec[['PhoneService', 'MultipleLines']].dropna()           # filas con las dos columnas
verificar(((tel['PhoneService'] == 'No') == (tel['MultipleLines'] == 'No phone service')).all(),   # sin teléfono <=> No phone service
          'telefonía: "No phone service" si y solo si PhoneService = No')   # regla de telefonía
inet = df_rec.dropna(subset=['InternetService'])                   # filas con InternetService
nis = (inet[ADICIONALES] == 'No internet service')                 # dónde dice "No internet service"
verificar(((inet['InternetService'] == 'No') == nis.any(axis=1)).all(),   # sin internet <=> algún No internet service
          'internet: "No internet service" si y solo si InternetService = No')   # regla de internet
verificar((nis.any(axis=1) == (nis | inet[ADICIONALES].isna()).all(axis=1)).all(),   # si uno lo dice, los demás también (o NaN)
          'los adicionales no mezclan "No internet service" con Yes/No')   # todos o ninguno
sin_tel = df_rec['PhoneService'] == 'No'                           # clientes sin teléfono
verificar(df_rec.loc[sin_tel, 'InternetService'].dropna().eq('DSL').all(),   # control de implementación de R8
          'oferta comercial (R8): todos los clientes sin teléfono tienen DSL')   # oferta comercial

OK - telefonía: "No phone service" si y solo si PhoneService = No
OK - internet: "No internet service" si y solo si InternetService = No
OK - los adicionales no mezclan "No internet service" con Yes/No
OK - oferta comercial (R8): todos los clientes sin teléfono tienen DSL


## 7.2 Rangos de las numéricas

R1, R6 y R7 escribieron valores numéricos nuevos. Controlamos que `tenure` siga siendo entera y entre 0 y 72, que `TotalCharges` valga 0 solo en los clientes con antigüedad 0 y que todos los cargos mensuales sean positivos. Esperamos tres `OK`.

In [21]:
# rangos de las numéricas después de R1, R6 y R7
verificar(df_rec['tenure'].dropna().between(0, 72).all() and (df_rec['tenure'].dropna() % 1 == 0).all(),   # rango y meses enteros
          'tenure sigue siendo entera y entre 0 y 72')             # antigüedad válida
con_ambos = df_rec.dropna(subset=['tenure', 'TotalCharges'])       # filas con antigüedad y total
verificar(((con_ambos['tenure'] == 0) == (con_ambos['TotalCharges'] == 0)).all(),   # total 0 <=> antigüedad 0
          'TotalCharges = 0 si y solo si tenure = 0 (solo los clientes nuevos)')   # coherencia de R1
verificar((df_rec['MonthlyCharges'].dropna() > 0).all(), 'todos los cargos mensuales son positivos')   # montos válidos

OK - tenure sigue siendo entera y entre 0 y 72
OK - TotalCharges = 0 si y solo si tenure = 0 (solo los clientes nuevos)
OK - todos los cargos mensuales son positivos


## 7.3 Cargos de R6 fuera del rango observado

R6 divide el total facturado por los meses de antigüedad, así que estima el cargo mensual promedio de toda la vida del cliente. Si el precio cambió, ese promedio puede caer un poco fuera del rango de los cargos actuales. Medimos cuántos casos hay y cuánto se alejan.

In [22]:
# R6 puede dar cargos un poco fuera del rango observado: son cargos promedio de toda la vida del cliente
rango = (df['MonthlyCharges'].min(), df['MonthlyCharges'].max())   # rango en los datos originales
nuevos = df_rec.loc[df['MonthlyCharges'].isna() & df_rec['MonthlyCharges'].notna(), 'MonthlyCharges']   # los que completó R6
fuera = nuevos[(nuevos < rango[0]) | (nuevos > rango[1])]          # los que caen fuera de ese rango
print(f'Rango observado de MonthlyCharges: {rango[0]} a {rango[1]}')   # muestro el resultado
print(f'Cargos de R6 fuera de ese rango: {len(fuera)} de {len(nuevos)}, entre {nuevos.min():.2f} y {nuevos.max():.2f}')   # muestro el resultado

Rango observado de MonthlyCharges: 18.7 a 118.75
Cargos de R6 fuera de ese rango: 30 de 1039, entre 13.92 y 120.62


De los 1.039 cargos que completa R6, 30 caen fuera del rango observado (18,70 a 118,75); el conjunto completado va de 13,92 a 120,62. No los recortamos. Son promedios de la vida del cliente, que difieren del cargo actual si el precio cambió, y quedan cerca de los extremos.

## 7.4 Faltantes que quedan

Lo que las reglas no completan pasa como `NaN` al notebook de machine learning, que lo imputa dentro del pipeline. Armamos una tabla con los faltantes de cada variable antes y después de las reglas, y el porcentaje que queda en train y en test. La partición es un sorteo estratificado que no mira los faltantes, así que esos porcentajes tendrían que ser parecidos, sea cual sea el mecanismo de pérdida. Esperamos 15 variables con faltantes antes de las reglas y un total que baje de 13.959 a 10.561.

In [23]:
faltantes = pd.DataFrame({                                         # NaN por columna, antes y después, y por conjunto
    'NaN antes': df[COLS_CAT + COLS_NUM].isna().sum(),             # antes de las reglas (incluye los blancos de TotalCharges)
    'NaN después': df_rec[COLS_CAT + COLS_NUM].isna().sum(),       # después de las reglas
    '% en train': df_rec.loc[df_rec['conjunto'] == 'train', COLS_CAT + COLS_NUM].isna().mean() * 100,   # % que falta en train
    '% en test': df_rec.loc[df_rec['conjunto'] == 'test', COLS_CAT + COLS_NUM].isna().mean() * 100,     # % que falta en test
})                                                                 # cierro la tabla de faltantes
faltantes['% antes'] = df[COLS_CAT + COLS_NUM].isna().mean() * 100   # % que faltaba antes de las reglas
faltantes = faltantes[faltantes['NaN antes'] > 0].sort_values('NaN después', ascending=False)   # solo columnas con NaN
print(f'Variables con faltantes antes de las reglas: {len(faltantes)}')   # cuántas columnas tenían NaN
print(f'Total de NaN: {faltantes["NaN antes"].sum()} -> {faltantes["NaN después"].sum()}')   # muestro el resultado
completas = df_rec[COLS_CAT + COLS_NUM].notna().all(axis=1).mean() * 100   # % de filas completas
print(f'Clientes con la fila completa: {df[COLS_CAT + COLS_NUM].notna().all(axis=1).mean() * 100:.1f}% -> {completas:.1f}%')   # muestro el resultado
faltantes.round(1)                                                 # muestro la tabla

Variables con faltantes antes de las reglas: 15
Total de NaN: 13959 -> 10561
Clientes con la fila completa: 12.0% -> 21.1%


,NaN antes,NaN después,% en train,% en test,% antes
Contract,1197,1197,16.9,17.5,17.0
PaperlessBilling,1197,1197,16.8,17.6,17.0
PaymentMethod,1197,1197,16.8,17.7,17.0
gender,916,916,13.0,13.2,13.0
SeniorCitizen,916,916,12.9,13.4,13.0
Partner,916,916,13.2,12.2,13.0
Dependents,916,916,13.0,13.1,13.0
MultipleLines,916,841,11.9,12.1,13.0
OnlineBackup,916,738,10.5,10.4,13.0
OnlineSecurity,916,722,10.5,9.4,13.0


Antes de las reglas faltan valores en 15 variables: las 14 que tienen faltantes en el CSV, con entre el 13% y el 17% cada una, y `TotalCharges`, por los 11 blancos convertidos a `NaN`. Las reglas bajan el total de 13.959 a 10.561 (el 24,3% recuperado) y los clientes con la fila completa pasan del 12,0% al 21,1%. Los porcentajes que quedan son parecidos en train y en test. Lo garantiza la partición estratificada al azar, que reparte los faltantes por igual entre los dos conjuntos sea cual sea su mecanismo. Las variables de cuenta y las demográficas no aparecen en ninguna regla, por eso conservan todos sus faltantes.

## 7.5 Faltantes en las columnas de servicios

Las reglas ya completaron los servicios de los clientes que no los tienen. Por lo tanto, los faltantes que quedan en `InternetService`, `MultipleLines` y los adicionales tendrían que ser todos de clientes que sí tienen el servicio. Lo controlamos porque define qué imputación es coherente: a esos clientes no se les puede asignar "No internet service" ni "No phone service".

Esto cambia el mecanismo de estos faltantes. En los datos originales el EDA no encontró evidencia en contra de MCAR (decisión 6). Después de las reglas, que un servicio falte depende de una columna observada, tener el servicio, así que los faltantes que quedan son MAR. El EDA lo mostró en su sección 6.5.

Que sean MAR no invalida imputarlos dentro del pipeline. Con MAR el mecanismo de pérdida se puede ignorar, porque depende solo de lo observado (Murphy, 2022, sec. 1.5.5). Además, el notebook de machine learning los imputa con la categoría "Desconocido" (decisión ML4), que nunca asigna "sin servicio" a un cliente que lo tiene. Esperamos tres `OK`.

In [24]:
# los NaN que quedan en columnas de servicios son de clientes que SÍ tienen el servicio
falta_inet = df_rec['InternetService'].isna()                      # sin dato de internet
verificar(not (df_rec.loc[falta_inet, ADICIONALES] == 'No internet service').any(axis=1).any()   # ningún adicional dice No internet service
          and not (df_rec.loc[falta_inet, 'PhoneService'] == 'No').any(),   # y ninguno es un cliente sin teléfono (R8)
          'los InternetService que faltan son de clientes con internet (DSL o fibra)')   # R4 y R8 ya actuaron
verificar(not (df_rec.loc[df_rec['MultipleLines'].isna(), 'PhoneService'] == 'No').any(),   # ninguno sin teléfono
          'los MultipleLines que faltan son de clientes con teléfono')   # R3 ya actuó
falta_adicional = df_rec[ADICIONALES].isna().any(axis=1)           # clientes con algún adicional faltante
verificar(not (df_rec.loc[falta_adicional, 'InternetService'] == 'No').any(),   # ninguno sin internet
          'los adicionales que faltan son de clientes con internet')   # R5 ya actuó

OK - los InternetService que faltan son de clientes con internet (DSL o fibra)
OK - los MultipleLines que faltan son de clientes con teléfono
OK - los adicionales que faltan son de clientes con internet


## 7.6 Lo que haría el imputador del pipeline

El imputador por moda del pipeline completa cada categórica con su valor más frecuente en train. Calculamos esa moda en las columnas de servicios que todavía tienen faltantes y controlamos que no sea una categoría de "sin servicio". Esperamos cuatro modas y un `OK`.

In [25]:
# la moda de train (la que usaría el imputador) no puede ser una categoría de "sin servicio"
servicios_nan = [c for c in ['InternetService', 'MultipleLines'] + ADICIONALES if df_rec[c].isna().any()]   # servicios con NaN
modas = df_rec.loc[df_rec['conjunto'] == 'train', servicios_nan].mode().iloc[0]   # moda de cada uno en train
print('Moda en train:', modas.to_dict())                           # muestro las modas
verificar(not modas.isin(['No internet service', 'No phone service']).any(),   # ninguna moda es sin servicio
          'imputar con la moda de train no crea filas incoherentes en los servicios')   # coherencia de la imputación

Moda en train: {'InternetService': 'Fiber optic', 'MultipleLines': 'No', 'OnlineSecurity': 'No', 'OnlineBackup': 'No'}
OK - imputar con la moda de train no crea filas incoherentes en los servicios


Los faltantes que quedan en `InternetService`, `MultipleLines` y los adicionales son todos de clientes que tienen el servicio, porque las reglas ya completaron a los que no lo tenían. Imputarles "No internet service" generaría filas incoherentes. Las modas de train ("Fiber optic", "No", "No", "No") y la categoría "Desconocido" que el EDA propone comparar (decisión 8) no tienen ese problema.

En las numéricas, la imputación por mediana tiene una limitación. R6 y R7 deducen el cargo o la antigüedad a partir del otro, así que no pueden actuar cuando faltan los dos. Por eso los clientes sin `tenure` también tienen `MonthlyCharges` faltante. Si el pipeline les imputa la mediana de cada una por separado, el producto `tenure × MonthlyCharges` deja de aproximar a `TotalCharges`, que en ellos sí está.

La celda siguiente cuenta esos clientes y muestra el producto que darían las medianas de train, sin modificar los datos. Esperamos 157 clientes, el 2,2% del total, y un producto lejos del `TotalCharges` real en la mayoría.

In [26]:
# los clientes sin tenure tampoco tienen MonthlyCharges: la mediana se imputaría en las dos a la vez (solo lectura)
sin_tenure = df_rec['tenure'].isna()                               # antigüedades que R7 no pudo deducir
ambos = sin_tenure & df_rec['MonthlyCharges'].isna()               # de esos, los que tampoco tienen cargo mensual
print(f'Clientes sin tenure: {sin_tenure.sum()}; de ellos, sin MonthlyCharges: {ambos.sum()} '   # coinciden todos
      f'({ambos.mean() * 100:.1f}% de los {len(df_rec)} clientes)')   # qué parte del dataset afecta
medianas = df_rec.loc[df_rec['conjunto'] == 'train', ['tenure', 'MonthlyCharges']].median()   # medianas de train, solo para mostrar el efecto
producto = medianas['tenure'] * medianas['MonthlyCharges']        # TotalCharges que implicarían las dos medianas
tc_real = df_rec.loc[ambos, 'TotalCharges']                        # TotalCharges registrado de esos clientes
lejos = ((producto - tc_real).abs() > 0.5 * tc_real).mean() * 100  # % en que el producto se aleja más de un 50%
print(f'Medianas de train: tenure {medianas["tenure"]:.0f}, MonthlyCharges {medianas["MonthlyCharges"]:.2f}; '   # valores que imputaría el pipeline
      f'producto {producto:.2f} para todos')                       # el mismo para los 157
print(f'TotalCharges real de esos clientes: de {tc_real.min():.2f} a {tc_real.max():.2f}; '   # rango real
      f'el producto se aleja más de un 50% en el {lejos:.1f}% de ellos')   # cuánto se rompe la relación

Clientes sin tenure: 157; de ellos, sin MonthlyCharges: 157 (2.2% de los 7043 clientes)
Medianas de train: tenure 29, MonthlyCharges 70.60; producto 2047.40 para todos
TotalCharges real de esos clientes: de 19.65 a 8670.10; el producto se aleja más de un 50% en el 72.6% de ellos


Los 157 clientes sin `tenure` tienen también `MonthlyCharges` faltante. Con las medianas de train, todos recibirían el mismo producto, lejos de su `TotalCharges` real en la mayoría. En el 2,2% de los clientes, entonces, la imputación rompe la relación `TotalCharges ≈ tenure × MonthlyCharges`.

Reconocemos esta limitación y la aceptamos por tres razones. Afecta a un porcentaje chico de los clientes. La imputación queda dentro del pipeline, ajustada solo con train, así que no filtra información del test. Y el modelo final es una regresión logística regularizada, lineal en sus variables, en la que unos pocos valores típicos pesan poco en los coeficientes.

> **Decisión M5:** no aplicar en este notebook ningún paso stateful ni ninguna transformación que el notebook de machine learning tenga que comparar por validación cruzada. Los faltantes que quedan pasan como `NaN`.
>
> **Justificación:** imputar, codificar, escalar y el PCA son stateful y, como explica la sección 6.1, van dentro del pipeline, ajustados solo con train. Ajustarlos aquí impediría además que la validación cruzada los reajuste en cada fold (Géron, 2022, cap. 2). También dejamos fuera tres transformaciones stateless que el EDA dejó para comparar por validación cruzada: recodificar "No internet service" (decisión 11), las features derivadas (decisión 15) y quitar `gender` y `PhoneService` (decisión 17). Aplicarlas aquí impediría esa comparación; además, `n_adicionales` necesita los adicionales ya imputados.

# 8. Guardado del dataset

## 8.1 Escritura

Guardamos un único CSV con una fila por cliente y las columnas en este orden:

- `customerID`, para rastrear cada fila hasta el CSV original (no es variable del modelo);
- `conjunto`, con la partición de la decisión 4;
- las 16 categóricas y las 3 numéricas, con el formato corregido y los faltantes reconstruidos;
- `Churn`, sin transformar.

No guardamos el índice de pandas, que solo numera las filas. El fin de línea es `\n` en cualquier sistema operativo, para que la huella del archivo dependa solo de los datos y no de si se escribió en Windows o en otro sistema. Esperamos 7.043 filas y 22 columnas.

In [27]:
COLUMNAS_SALIDA = [COL_ID, 'conjunto'] + COLS_CAT + COLS_NUM + [TARGET]   # orden de las columnas
salida = df_rec[COLUMNAS_SALIDA].reset_index(drop=True)            # tabla a guardar
salida.to_csv(RUTA_SALIDA, index=False, lineterminator='\n')     # sin el índice y con fin de línea \n en cualquier sistema
print('Guardado:', RUTA_SALIDA, '| shape:', salida.shape)          # ruta y tamaño del archivo escrito

Guardado: data/telco_manipulado.csv | shape: (7043, 22)


## 8.2 Lectura de control

Un CSV puede alterar datos sin aviso, por ejemplo con un decimal que se redondea o un texto vacío que se lee como `NaN`. Releemos el archivo como lo va a leer el notebook de machine learning y lo comparamos celda por celda con lo que escribimos. Si algo difiere, el mensaje de error incluye el detalle que da pandas: la columna y los valores distintos. `check_exact=True` hace que no se tolere ninguna diferencia de redondeo. Con `check_dtype=False` no exigimos el mismo tipo de dato en memoria, porque lo que importa es que los valores sean iguales. Esperamos un `OK`.

In [28]:
leido = pd.read_csv(RUTA_SALIDA)                                   # lo leo como lo va a leer el notebook de ML
try:                                                               # comparo celda por celda
    pd.testing.assert_frame_equal(leido, salida, check_dtype=False, check_exact=True)   # mismos valores exactos y mismos NaN
    identico = True                                                # no hubo diferencias
    detalle = ''                                                   # sin diferencias, el mensaje queda como siempre
except AssertionError as error:                                    # alguna celda difiere
    identico = False                                               # el control va a fallar
    detalle = f'. Diferencia encontrada: {error}'                  # guardo qué columna y qué valores difieren
verificar(identico, f'{RUTA_SALIDA} se lee idéntico a lo que se guardó ({leido.shape[0]} filas x {leido.shape[1]} columnas)'   # ida y vuelta
          + detalle)                                               # si falla, el mensaje incluye el detalle de pandas

OK - data/telco_manipulado.csv se lee idéntico a lo que se guardó (7043 filas x 22 columnas)


## 8.3 Huella digital

La huella SHA-256 es un resumen de 64 caracteres hexadecimales que cambia si cambia un solo byte del archivo. Mostramos los primeros 16 y los comparamos con `HUELLA_REGISTRADA`. Con las versiones de `requirements.txt` deberían coincidir. Si difieren, el notebook avisa y sigue, sin detenerse, porque el archivo ya pasó los controles; conviene revisar las versiones antes de entrenar.

In [29]:
with open(RUTA_SALIDA, 'rb') as archivo:                          # abro el archivo en binario y lo cierro al terminar
    huella = hashlib.sha256(archivo.read()).hexdigest()[:16]       # huella digital del archivo
print('Huella SHA-256 (primeros 16 caracteres):', huella)          # sirve para comparar entre computadoras
if huella == HUELLA_REGISTRADA:                                    # mismo archivo que al preparar el trabajo
    print('La huella coincide con la registrada:', HUELLA_REGISTRADA)   # resultado reproducido
else:                                                              # algo cambió en los datos o en las versiones
    print('AVISO: la huella difiere de la registrada', HUELLA_REGISTRADA, '- revisar versiones de requirements.txt')   # aviso sin detener

Huella SHA-256 (primeros 16 caracteres): b61cf37c562dfe9e
La huella coincide con la registrada: b61cf37c562dfe9e


## 8.4 Lo que recibe el notebook de machine learning

Cerramos con una tabla que muestra el rol de cada columna y cuántos faltantes tiene en train y en test. Esperamos `customerID`, `conjunto` y `Churn` completos.

In [30]:
# resumen de lo que recibe el notebook de machine learning
resumen = pd.DataFrame({                                           # tabla con una fila por columna del CSV
    'tipo': ['identificador', 'partición'] + ['categórica'] * len(COLS_CAT) + ['numérica'] * len(COLS_NUM) + ['objetivo'],   # rol
    'NaN en train': leido[leido['conjunto'] == 'train'].isna().sum().values,   # faltantes que imputa el pipeline
    'NaN en test': leido[leido['conjunto'] == 'test'].isna().sum().values,     # ídem en test
}, index=leido.columns)                                            # una fila por columna
resumen                                                            # muestro la tabla

,tipo,NaN en train,NaN en test
customerID,identificador,0,0
conjunto,partición,0,0
gender,categórica,730,186
SeniorCitizen,categórica,727,189
Partner,categórica,744,172
Dependents,categórica,731,185
PhoneService,categórica,43,11
MultipleLines,categórica,671,170
InternetService,categórica,523,113
OnlineSecurity,categórica,589,133


`data/telco_manipulado.csv` tiene 7.043 filas y 22 columnas y se lee idéntico a lo que guardamos. Su huella empieza con `b61cf37c562dfe9e` y coincide con la registrada. `customerID`, `conjunto` y `Churn` están completos, y los `NaN` que quedan en las variables son los que imputa el pipeline.

> **Decisión M6:** guardar un solo archivo, `data/telco_manipulado.csv`, con el identificador, la partición, las 19 variables y el objetivo; verificar que se lea idéntico y registrar su huella digital.
>
> **Justificación:** un archivo único con la columna `conjunto` mantiene juntas cada fila y su partición, de modo que train y test no pueden desincronizarse como podría ocurrir con dos archivos. La partición queda estable entre corridas (Géron, 2022, cap. 2). El CSV es un formato abierto que se lee con cualquier versión de pandas. La lectura de control y la huella hacen verificable que el notebook de machine learning parte exactamente de estos datos.

El notebook de machine learning lee el archivo así:

```python
datos = pd.read_csv('data/telco_manipulado.csv')
train = datos[datos['conjunto'] == 'train']
test = datos[datos['conjunto'] == 'test']
X_train = train.drop(columns=['customerID', 'conjunto', 'Churn'])
y_train = (train['Churn'] == 'Yes').astype(int)
```

# 9. Registro de decisiones, herencia y ejecución

## 9.1 Decisiones de este notebook

| # | Decisión | Justificación (síntesis) | Decisión del EDA |
|---|---|---|---|
| M1 | Validar la estructura del CSV antes de transformarlo y detenerse si algo no coincide | Los supuestos de las reglas se vuelven controles; el error aparece donde se origina | 1 y 2 |
| M2 | Corregir el formato con `corregir_formato` y controlar faltantes, dominio y rangos | Transformaciones stateless aplicadas siempre igual, sin errores silenciosos | 1, 2 y 3 |
| M3 | Leer la partición del EDA, controlarla y unirla como columna `conjunto`; recrearla si falta | El mismo test en los tres notebooks, sin depender de la versión de scikit-learn | 4 |
| M4 | Aplicar R1 a R9 a todo el dataset y probar que son stateless | 3.398 valores recuperados; mismo resultado sobre todo el dataset, por conjunto y cliente por cliente | 6 y 7 |
| M5 | No aplicar pasos stateful ni alternativas que se comparan por validación cruzada | Evita la fuga y deja la comparación al notebook de machine learning | 8, 10, 11, 12, 15, 16 y 17 |
| M6 | Guardar un solo CSV con ID, partición, variables y objetivo; verificar la lectura y registrar la huella | Fila y partición siempre juntas; resultado verificable en otra computadora | 4 |

## 9.2 Qué hereda el notebook de machine learning

`clusterai_guillermo_inostroza_machine_learning.ipynb` parte de `data/telco_manipulado.csv` y hace dentro del pipeline, ajustando solo con train en cada fold, lo que el EDA dejó para esa etapa:

| Paso | Decisión del EDA | Por qué va en el pipeline |
|---|---|---|
| Imputar los `NaN` que quedan: mediana en numéricas; moda contra "Desconocido" en categóricas | 8 | Aprende la mediana o la moda (stateful) |
| Escalar las numéricas con `StandardScaler`; probar `log1p(TotalCharges)` | 10 | Aprende la media y el desvío (stateful) |
| One-hot de las categóricas; comparar recodificar "No internet service" | 11 | Aprende las categorías; la recodificación se compara por validación cruzada |
| Features derivadas (`n_adicionales`, tramos de `tenure`) | 15 | Se comparan por validación cruzada; `n_adicionales` necesita los adicionales imputados |
| PCA después de imputar, codificar y escalar | 16 | Aprende las componentes (stateful) |
| Probar sin `gender` y `PhoneService`, y con y sin `TotalCharges` | 12 y 17 | Se decide por validación cruzada |
| Elegir modelos por ROC-AUC con validación cruzada estratificada; umbral por costos; test una sola vez | 13, 14 y 18 | Es la evaluación |
| No eliminar filas con `dropna`, no agregar indicadoras de faltante, no quitar outliers | 5, 6 y 9 | Criterio general, sin paso de código |

## 9.3 Cómo ejecutar los notebooks en otra computadora

1. Usar Python 3.11 o posterior (pandas 3 no se instala en versiones anteriores) e instalar las versiones fijadas con `pip install -r requirements.txt`. Los notebooks se ejecutaron con Python 3.14.
2. Abrir los notebooks desde la carpeta `tp_final_telco/`, que contiene `requirements.txt` y la carpeta `data/` con `telco_churn_clusterai.csv`. Todas las rutas son relativas a esa carpeta.
3. Ejecutar los notebooks en orden: `clusterai_guillermo_inostroza_eda.ipynb`, que crea `data/particion_train_test.csv`; este notebook, que crea `data/telco_manipulado.csv`; y `clusterai_guillermo_inostroza_machine_learning.ipynb`. Si este notebook se ejecuta sin haber corrido el EDA, recrea la partición con el mismo criterio y lo avisa.
4. Ejecutar todas las celdas de arriba hacia abajo. Cada control tiene que imprimir `OK`; si uno falla, el notebook se detiene. La huella del archivo de salida debería empezar con `b61cf37c562dfe9e`. Si no, el notebook solo avisa y sigue, y conviene revisar las versiones antes de entrenar.

# 10. Referencias

**Bibliografía**

- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3.ª ed.). O'Reilly. Capítulo 2.
- James, G., Witten, D., Hastie, T., Tibshirani, R. y Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer. Secciones 5.1 y 12.3.
- Murphy, K. P. (2022). *Probabilistic Machine Learning: An Introduction*. MIT Press. Sección 1.5.5.
- VanderPlas, J. (2016). *Python Data Science Handbook*. O'Reilly. Capítulo 3.

**Trabajo propio**

- Notebook de EDA de este trabajo (`clusterai_guillermo_inostroza_eda.ipynb`), con la evidencia de las decisiones 1 a 18.